# Introduction to MONAI

MONAI is a deep learning library for medical imaging research. It provides a comprehensive set of tools and utilities for building, training, and evaluating deep learning models in the medical imaging domain.

## Key Features

1. **Data Loading and Preprocessing**: MONAI offers a wide range of data loading and preprocessing functionalities, including support for various image formats, normalization, and augmentation techniques.
2. **Transformations**: It includes a rich set of transformations for data augmentation, normalization, and other preprocessing steps.
3. **Model Building**: MONAI provides a modular approach to building deep learning models, including popular architectures like U-Net, V-Net, and Residual Networks.
4. **Training and Evaluation**: It offers tools for training models, including loss functions, optimizers, and evaluation metrics, making it easier to develop and deploy medical image segmentation models.
5. **Integration with PyTorch**: MONAI is built on top of PyTorch, allowing for seamless integration with the PyTorch ecosystem and leveraging its powerful ecosystem of tools and libraries.

## Installation

```bash
pip install monai
```

We have already installed MONAI in your DL_labs_GPU environment. We will do the same as we did in PyTorch in the notebook "Segmentation and UNET". The idea is you will see the difference between the two frameworks and how MONAI may simplify the process of building and training deep learning models for medical imaging tasks.

In [ ]:
# Some imports

import monai
import torch
from torch.utils.data import DataLoader
from monai.transforms import (
    EnsureChannelFirstd,
    AsDiscreted,
    Compose,
    LoadImaged,
    Orientationd,
    Randomizable,
    Resized,
    ScaleIntensityd,
    Spacingd,
    EnsureTyped,
    Lambda
)
import monai.metrics as metrics
import os
import tempfile
from cm2003.decathlon_dataset import get_decathlon_dataloader
from cm2003.unet import UNET
from cm2003.train import train

import matplotlib.pyplot as plt


In [ ]:

root_dir = './cm2003/datasets'
task = "Task04_Hippocampus"

train_loader = get_decathlon_dataloader(root_dir, task, "training", batch_size=4, num_workers=2, shuffle=True)
val_loader = get_decathlon_dataloader(root_dir, task, "validation", batch_size=4, num_workers=2, shuffle=False)




In [ ]:
# Load UNET model from MONAI

model = monai.networks.nets.UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=3,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
)

# Define loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Define device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

Implement the training loop for the model with the same structure as for PyTorch. You may even make use of the train function we wrote in PyTorch in `cm2003/train.py`.

In [ ]:
from cm2003.train import train
# Your code here
trained_model = None



Now let's evaluate the model on the validation set with the metrics in MONAI. We give the example with Hausdorff Distance but you should try to find other metrics in the documentation: [https://docs.monai.io/en/stable/metrics.html](https://docs.monai.io/en/stable/metrics.html) appropriate for the segmentation task. We can interpret the Hausdorff Distance as the maximum difference between the predicted segmentation boundary and the true boundary for 95% of the points. In this task, we are trying to segment the hippocampus, which is a small structure and the unit of the distance is millimeters.

In [ ]:
import torch
from monai.metrics import HausdorffDistanceMetric

hausdorff_distance = HausdorffDistanceMetric(include_background=False, reduction="mean", percentile=95)

for val_data in val_loader:
    val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
    val_outputs = torch.nn.functional.one_hot(trained_model(val_inputs).argmax(dim=1), num_classes=3).permute(0, 3, 1, 2)
    val_labels = torch.nn.functional.one_hot(val_labels.long(), num_classes=3).squeeze(1).permute(0, 3, 1, 2)
    
    # Identify valid classes
    valid_classes = torch.logical_and(torch.any(val_labels, dim=(0, 2, 3)), torch.any(val_outputs, dim=(0, 2, 3)))
    
    # Only compute Hausdorff distance for valid classes
    if torch.any(valid_classes):
        hausdorff_distance(val_outputs[:, valid_classes], val_labels[:, valid_classes])
    else:
        print("No valid classes found for Hausdorff distance calculation.")

result = hausdorff_distance.aggregate().item()
print(f'95th percentile Hausdorff distance: {result:.4f}')

## Evaluation for two other metrics

**Implement evaluation for two other metrics in MONAI and comment on the results.**

Now we have trained and evaluated the model, let's change the loss function to DiceLoss and train the model again which is another loss function commonly used in medical image segmentation. Look at the documentation of MONAI to find out how to do this: [https://docs.monai.io/en/stable/losses.html](https://docs.monai.io/en/stable/losses.html)

After you have trained the model, you can evaluate the model on the validation set with the new loss function.

In [ ]:
# Your code here

## Optional part

MONAI also provides implementations of many networks apart from UNET. You can find them in the `monai.networks` module, see [https://docs.monai.io/en/stable/networks.html](https://docs.monai.io/en/stable/networks.html). Try to use one of them for the Hippocampus dataset and evaluate the performance. Compare the results with UNET and DiceLoss. 

In [ ]:
# Your code here